# Synthesis and Policy Implications

This notebook pulls together the strongest results from diagnostics, validation, and interpretation into a compact thesis-ready summary.

It is intended to answer three questions:
- What do the time-series diagnostics imply about PHP FX behavior?
- Why do hybrid models make sense statistically and economically?
- What should policymakers and risk managers take away?

In [8]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')
from statistical_validation import load_config, get_project_paths, discover_forecasts

config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
paths = get_project_paths(config)
forecasts = discover_forecasts(config)

eval_dir = paths['results_dir'] / 'evaluation'
diag_dir = paths['results_dir'] / 'diagnostics'
print('Target:', active_target)
print('Evaluation dir:', eval_dir)
print('Diagnostics dir:', diag_dir)

Target: PHP
Evaluation dir: results\PHP\evaluation
Diagnostics dir: results\PHP\diagnostics


#### Interpretation
This setup and load stage consolidates all key outputs used in final synthesis. If any source is missing here, policy conclusions should be treated as provisional.

In [9]:
metrics_path = eval_dir / 'metrics_summary.csv'
dm_path = eval_dir / 'dm_test_mse.csv'
stationarity_path = diag_dir / 'stationarity_panel.csv'
breaks_path = diag_dir / 'structural_breaks_panel.csv'

metrics_df = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
dm_df = pd.read_csv(dm_path) if dm_path.exists() else pd.DataFrame()
stationarity_df = pd.read_csv(stationarity_path) if stationarity_path.exists() else pd.DataFrame()
breaks_df = pd.read_csv(breaks_path) if breaks_path.exists() else pd.DataFrame()

summary = pd.DataFrame({
    'Metric': [
        'Test models discovered',
        'Pairs covered',
        'DM tests with p<0.05 share',
        'Stationarity rows available',
        'Structural-break rows available',
    ],
    'Value': [
        forecasts['Model'].nunique(),
        forecasts['Pair'].nunique(),
        float((dm_df['p_value'] < 0.05).mean()) if not dm_df.empty else np.nan,
        len(stationarity_df),
        len(breaks_df),
    ]
})
display(summary)

,Metric,Value
0,Test models discovered,15.00
1,Pairs covered,5.00
2,DM tests with p<0.05 share,0.77
3,Stationarity rows available,10.00
4,Structural-break rows available,5.00


#### Interpretation
The summary table aggregates evidence from accuracy, diagnostics, and robustness. Read this as the final quantitative basis for selecting preferred model classes.

## Multi-Horizon Performance Check (h=1,5,10)

This section compares pseudo multi-horizon error profiles to show where hybrid and linear families diverge most.

In [10]:
mh = forecasts[forecasts['Set'] == 'test'].copy()
mh['Date'] = pd.to_datetime(mh['Date'])
mh = mh.sort_values(['Pair', 'Model', 'Date'])

def horizon_panel(df, h):
    out = []
    for (pair, model), sub in df.groupby(['Pair', 'Model']):
        sub = sub.sort_values('Date').copy()
        y_true_h = sub['Actual'].rolling(h).sum().shift(-(h - 1))
        y_pred_h = sub['Forecast'].rolling(h).sum().shift(-(h - 1))
        work = pd.DataFrame({'Date': sub['Date'], 'Actual_h': y_true_h, 'Forecast_h': y_pred_h}).dropna()
        if work.empty:
            continue
        err = work['Actual_h'] - work['Forecast_h']
        out.append({
            'Pair': pair,
            'Model': model,
            'Horizon': h,
            'MAE_h': float(err.abs().mean()),
            'RMSE_h': float(np.sqrt(np.mean(np.square(err)))),
            'n_obs': int(len(work)),
        })
    return pd.DataFrame(out)

mh_frames = [horizon_panel(mh, h) for h in [1, 5, 10]]
mh_df = pd.concat([x for x in mh_frames if not x.empty], ignore_index=True) if any(not x.empty for x in mh_frames) else pd.DataFrame()

if mh_df.empty:
    print('No multi-horizon panel could be built from test forecasts.')
else:
    mh_df['Class'] = np.where(mh_df['Model'].str.startswith('hybrid_'), 'hybrid', 'linear_or_baseline')
    display(mh_df.sort_values(['Horizon', 'Pair', 'RMSE_h']).head(40))

    class_perf = (
        mh_df.groupby(['Pair', 'Horizon', 'Class'], as_index=False)
        .agg(MAE_h=('MAE_h', 'mean'), RMSE_h=('RMSE_h', 'mean'))
    )
    pivot = class_perf.pivot_table(index=['Pair', 'Horizon'], columns='Class', values='RMSE_h').reset_index()
    if {'hybrid', 'linear_or_baseline'}.issubset(pivot.columns):
        pivot['Hybrid_minus_Linear_RMSE'] = pivot['hybrid'] - pivot['linear_or_baseline']
        divergence = pivot.sort_values('Hybrid_minus_Linear_RMSE')
        display(divergence)
    else:
        divergence = pd.DataFrame()

    mh_path = eval_dir / f'multi_horizon_summary_{active_target}.csv'
    mh_df.to_csv(mh_path, index=False)
    print('Saved:', mh_path)
    if not divergence.empty:
        div_path = eval_dir / f'multi_horizon_divergence_{active_target}.csv'
        divergence.to_csv(div_path, index=False)
        print('Saved:', div_path)

,Pair,Model,Horizon,MAE_h,RMSE_h,n_obs,Class
7,CNYPHP_RET,hybrid_arimax_mlp,1,0.365362,0.529127,423,hybrid
8,CNYPHP_RET,hybrid_arimax_svr,1,0.369996,0.533952,423,hybrid
11,CNYPHP_RET,hybrid_varx_mlp,1,0.374367,0.535960,423,hybrid
12,CNYPHP_RET,hybrid_varx_svr,1,0.375845,0.537257,423,hybrid
5,CNYPHP_RET,hybrid_arima_mlp,1,0.377803,0.544252,423,hybrid
9,CNYPHP_RET,hybrid_var_mlp,1,0.387458,0.551099,423,hybrid
6,CNYPHP_RET,hybrid_arima_svr,1,0.382081,0.552395,423,hybrid
10,CNYPHP_RET,hybrid_var_svr,1,0.393824,0.561002,423,hybrid
1,CNYPHP_RET,arimax,1,0.379623,0.565585,423,linear_or_baseline
14,CNYPHP_RET,varx,1,0.383369,0.566618,423,linear_or_baseline


Class,Pair,Horizon,hybrid,linear_or_baseline,Hybrid_minus_Linear_RMSE
12,USDPHP_RET,1,0.522646,0.584388,-0.061742
3,HKDPHP_RET,1,0.522768,0.584402,-0.061634
9,SGDPHP_RET,1,0.526824,0.576524,-0.049700
0,CNYPHP_RET,1,0.543130,0.587724,-0.044593
6,JPYPHP_RET,1,0.734447,0.763878,-0.029431
14,USDPHP_RET,10,1.317442,1.342351,-0.024909
5,HKDPHP_RET,10,1.335942,1.348132,-0.012191
13,USDPHP_RET,5,0.997379,0.996664,0.000715
4,HKDPHP_RET,5,1.014509,1.002495,0.012013
2,CNYPHP_RET,10,1.365335,1.328694,0.036641


Saved: results\PHP\evaluation\multi_horizon_summary_PHP.csv
Saved: results\PHP\evaluation\multi_horizon_divergence_PHP.csv


## Draft Policy and Research Implications

- For BSP and FX risk managers: the combination of stationarity, breaks, and volatility clustering means a static linear rule is insufficient; hybrid correction is preferable.
- For model builders: add macro exogenous variables such as US rate surprises, VIX, trade indicators, and remittance proxies to test ARIMAX/VARX variants.
- For thesis framing: the project is not only a forecast comparison; it is evidence that PHP FX is governed by both linear memory and nonlinear shock transmission.
- For Vietnam policy relevance: similar USD/CNY transmission channels can propagate into VND pressure episodes through trade competitiveness, USD funding conditions, and regional risk re-pricing.

In [11]:
fig = px.bar(
    summary,
    x='Metric',
    y='Value',
    title='Synthesis summary of key notebook outputs',
    template='plotly_white'
)
fig.show()

print('Policy note: when DM significance is high and break/volatility evidence is persistent, a hybrid residual model is justified for operational PHP FX monitoring.')

Policy note: when DM significance is high and break/volatility evidence is persistent, a hybrid residual model is justified for operational PHP FX monitoring.


#### Interpretation
This figure communicates implications for decision-making under volatility and spillovers. Policy or hedging recommendations should prioritize models that remain stable under stress.